In [3]:
# %% [markdown]
# BEACON — Memory-safe incremental conversion (fixes the crash)
#
# Strategy: process ONE category at a time, write it straight to Parquet
# using Polars' streaming engine (never holds the full category in RAM,
# let alone all 9 categories at once). If this crashes partway through,
# already-converted categories are safe on disk — just skip them and
# resume from where it stopped.

# %%

import glob
import os
import polars as pl

NET_ROOT = r"D:\Malware Dataset\NetCSVs"
MEM_ROOT = r"D:\Malware Dataset\MemoryCSVs"
OUTPUT_DIR = r"D:\Malware Dataset\processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CATEGORIES = ["Rootkit", "Hoax"]

# %%
def load_with_metadata(filepath: str, category: str) -> pl.LazyFrame:
    sample_id = os.path.splitext(os.path.basename(filepath))[0]
    lf = pl.scan_csv(filepath, ignore_errors=True)
    cols = lf.collect_schema().names()
    exprs = [pl.lit(sample_id).alias("sample_id")]
    if "label" not in cols:
        exprs.append(pl.lit(category).alias("label"))
    return lf.with_columns(exprs)

# %%
# ============================================================
# Convert NETWORK data — one category at a time, streamed to disk
# ============================================================
for cat in CATEGORIES:
    out_path = os.path.join(OUTPUT_DIR, f"net_{cat}.parquet")

    if os.path.exists(out_path):
        print(f"[SKIP] {cat} already converted -> {out_path}")
        continue

    files = glob.glob(os.path.join(NET_ROOT, cat, "*.csv"))
    if not files:
        print(f"[WARN] No files found for {cat}")
        continue

    print(f"[RUNNING] {cat}: {len(files)} files...")
    lazy_frames = [load_with_metadata(f, cat) for f in files]
    combined = pl.concat(lazy_frames, how="diagonal_relaxed")

    # sink_parquet streams row-groups to disk instead of building the
    # full result in RAM first — this is the key fix for the crash
    combined.sink_parquet(out_path)
    print(f"[DONE] {cat} -> {out_path}")

print("\nAll network categories processed.")

# %%
# ============================================================
# Convert MEMORY data — same pattern
# ============================================================
for cat in CATEGORIES:
    out_path = os.path.join(OUTPUT_DIR, f"mem_{cat}.parquet")

    if os.path.exists(out_path):
        print(f"[SKIP] {cat} already converted -> {out_path}")
        continue

    files = glob.glob(os.path.join(MEM_ROOT, cat, "*.csv"))
    if not files:
        print(f"[WARN] No files found for {cat}")
        continue

    print(f"[RUNNING] {cat}: {len(files)} files...")
    lazy_frames = [load_with_metadata(f, cat) for f in files]
    combined = pl.concat(lazy_frames, how="diagonal_relaxed")
    combined.sink_parquet(out_path)
    print(f"[DONE] {cat} -> {out_path}")

print("\nAll memory categories processed.")

# %%
# ============================================================
# Cheap row counts — reads Parquet METADATA only, not the actual data
# ============================================================
import pyarrow.parquet as pq

print("Row counts (from metadata, no data read):")
for cat in CATEGORIES:
    for prefix in ["net", "mem"]:
        path = os.path.join(OUTPUT_DIR, f"{prefix}_{cat}.parquet")
        if os.path.exists(path):
            n = pq.ParquetFile(path).metadata.num_rows
            print(f"  {prefix}_{cat}: {n:,} rows")

[RUNNING] Rootkit: 250 files...
[DONE] Rootkit -> D:\Malware Dataset\processed\net_Rootkit.parquet
[RUNNING] Hoax: 250 files...
[DONE] Hoax -> D:\Malware Dataset\processed\net_Hoax.parquet

All network categories processed.
[RUNNING] Rootkit: 1058 files...
[DONE] Rootkit -> D:\Malware Dataset\processed\mem_Rootkit.parquet
[RUNNING] Hoax: 1039 files...
[DONE] Hoax -> D:\Malware Dataset\processed\mem_Hoax.parquet

All memory categories processed.
Row counts (from metadata, no data read):
  net_Rootkit: 66,857 rows
  mem_Rootkit: 1,120 rows
  net_Hoax: 54,586 rows
  mem_Hoax: 1,099 rows
